In [0]:
from pyspark.sql import SparkSession
from zipfile import ZipFile
from io import BytesIO
import datetime

spark = SparkSession.builder.getOrCreate()

# --- CONFIG ---
storage_account = "grtadlsdev"
container = "raw"
bronze_folder = "bronze"

# pick latest ZIP from raw
files = dbutils.fs.ls(f"abfss://{container}@{storage_account}.dfs.core.windows.net/")
zips = [f for f in files if f.name.startswith("gtfs_") and f.name.endswith(".zip")]
latest = sorted(zips, key=lambda x: x.modificationTime, reverse=True)[0].path

# timestamped output folder
run_date = datetime.datetime.now(datetime.UTC).strftime("%Y%m%d_%H%M%S")
out_path = f"abfss://{bronze_folder}@{storage_account}.dfs.core.windows.net/{run_date}/"

# read ZIP as bytes
binary_df = spark.read.format("binaryFile").load(latest)
zip_bytes = BytesIO(binary_df.collect()[0].content)

# unzip
with ZipFile(zip_bytes, 'r') as z:
    for name in z.namelist():
        content = z.read(name).decode("utf-8")
        (spark.createDataFrame([(content,)], ["value"])
             .write.mode("overwrite")
             .text(out_path + name))